<a href="https://colab.research.google.com/github/wangari2kimura/Special-topics-in-networking-/blob/main/CNS_4107_Lab3_Advanced_Network_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNS 4107: Special Topics in Computer Networks- Network Automation
## Lab 2: Advanced Network Automation
### Paramiko · Netmiko · NAPALM · Nornir · Ansible · REST APIs

---

**Objective:** By the end of this lab, you will be able to:
- Use **Paramiko** to build raw SSH connections and execute remote commands
- Use **Netmiko** to automate multi-vendor network devices over SSH
- Use **NAPALM** to retrieve and manage device state in a vendor-agnostic way
- Use **Nornir** to run automation tasks across a device inventory in parallel
- Use **Ansible** to push configurations declaratively via playbooks
- Consume real-world **REST APIs** for network intelligence

**Estimated Time:** 2 – 3 hours
**Prerequisites:** Lab 1 (Network Automation Intro), basic Python

---

> ⚠️ **Colab Note:** Real SSH requires hardware. This lab uses **simulated device environments** so every cell runs fully in Google Colab. Callouts marked 🔌 show the exact change for a real device.
>
> 💡 Run all cells top-to-bottom. Complete each **✏️ Exercise** before moving on.

---
## Part 0 — Environment Setup

In [1]:
!pip install paramiko netmiko napalm nornir nornir-utils requests ansible-core --quiet 2>&1 | tail -5

import os, json, time, datetime, re, yaml
from pathlib import Path
import requests

LAB_DIR = Path("/content/lab2")
LAB_DIR.mkdir(exist_ok=True)
print("✅ Environment ready.")
print(f"📁 Lab directory: {LAB_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 8.2 MB/s eta 0:00:00
✅ Environment ready.
📁 Lab directory: /content/lab2


---
## Part 1 — Paramiko: Raw SSH Automation

**Paramiko** is a pure-Python SSH implementation giving full, low-level control over SSH sessions — useful for custom auth flows, SFTP file transfers, and devices other tools don't support.

### SSH Session Flow
```
Client                          SSH Server (Network Device)
  |--- TCP connect (port 22) -------> |
  |--- SSH handshake / key exchange ->|
  |--- Authenticate (user/password) ->|
  |<-- Shell or exec channel opened --|
  |--- send command (e.g. show ver)  ->|
  |<-- receive output ----------------|
  |--- close channel / transport ----->|
```

### 1.1 Simulated SSH Environment
We build a `MockSSHClient` that behaves exactly like a real Paramiko session, so all code patterns are authentic.

In [2]:
# ── Simulated device command database ──────────────────────────
DEVICE_COMMAND_DB = {
    "show version": """\
Cisco IOS Software, Version 15.2(4)M7
Router uptime is 4 days, 7 hours, 12 minutes
cisco CISCO2901/K9 (revision 1.0) with 512000K/51200K bytes of memory.
2 Gigabit Ethernet interfaces
Configuration register is 0x2102""",

    "show ip interface brief": """\
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.0.0.1        YES NVRAM  up                    up
GigabitEthernet0/1     192.168.1.1     YES NVRAM  up                    up
Loopback0              1.1.1.1         YES NVRAM  up                    up
GigabitEthernet0/2     unassigned      YES NVRAM  administratively down down""",

    "show ip route": """\
Codes: C - connected, S - static
C    10.0.0.0/24 is directly connected, GigabitEthernet0/0
C    192.168.1.0/24 is directly connected, GigabitEthernet0/1
S*   0.0.0.0/0 [1/0] via 10.0.0.254""",

    "show running-config | include hostname": "hostname RTR-CORE-01",

    "show interfaces GigabitEthernet0/0": """\
GigabitEthernet0/0 is up, line protocol is up
  Hardware is CN Gigabit Ethernet, address is aabb.cc00.0100
  Internet address is 10.0.0.1/24
  MTU 1500 bytes, BW 1000000 Kbit/sec
  5 minute input rate 1245000 bits/sec, 240 packets/sec
  5 minute output rate 980000 bits/sec, 190 packets/sec"""
}


class _MockStream:
    def __init__(self, data): self._data = data.encode() if isinstance(data, str) else data
    def read(self): return self._data


class MockSSHClient:
    """Drop-in replacement for paramiko.SSHClient for Colab simulation."""
    def __init__(self):
        self.connected = False
        self._host = None

    def set_missing_host_key_policy(self, policy): pass

    def connect(self, hostname, port=22, username=None, password=None, timeout=10):
        time.sleep(0.2)
        self.connected = True
        self._host = hostname
        print(f"  🔐 [Paramiko] Connected to {hostname}:{port} as '{username}'")

    def exec_command(self, command):
        if not self.connected:
            raise RuntimeError("Not connected")
        output = DEVICE_COMMAND_DB.get(command.strip(), f"% Unknown command: '{command}'")
        return None, _MockStream(output), _MockStream("")

    def close(self):
        self.connected = False
        print(f"  🔒 [Paramiko] Connection to {self._host} closed")


print("✅ Mock SSH environment ready.")

✅ Mock SSH environment ready.


### 1.2 Connecting and Running Commands

In [3]:
import paramiko   # real import — installed above

def ssh_run_commands(host, username, password, commands, port=22):
    """
    Open an SSH session, run each command, return {command: output}.
    🔌 REAL DEVICE: replace MockSSHClient() with paramiko.SSHClient()
    """
    client = MockSSHClient()                          # <- swap for real hardware
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    results = {}
    try:
        client.connect(hostname=host, port=port,
                       username=username, password=password, timeout=10)
        for cmd in commands:
            _, stdout, stderr = client.exec_command(cmd)
            out = stdout.read().decode("utf-8", errors="replace").strip()
            err = stderr.read().decode("utf-8", errors="replace").strip()
            results[cmd] = out if out else err
            print(f"  ✅ Executed: {cmd!r}")
    finally:
        client.close()
    return results


ssh_output = ssh_run_commands(
    host="192.168.1.1", username="admin", password="cisco123",
    commands=["show version", "show ip interface brief", "show ip route"]
)

print("\n" + "="*55)
for cmd, out in ssh_output.items():
    print(f"\n>>> {cmd}\n{out}")

  🔐 [Paramiko] Connected to 192.168.1.1:22 as 'admin'
  ✅ Executed: 'show version'
  ✅ Executed: 'show ip interface brief'
  ✅ Executed: 'show ip route'
  🔒 [Paramiko] Connection to 192.168.1.1 closed


>>> show version
Cisco IOS Software, Version 15.2(4)M7
Router uptime is 4 days, 7 hours, 12 minutes
cisco CISCO2901/K9 (revision 1.0) with 512000K/51200K bytes of memory.
2 Gigabit Ethernet interfaces
Configuration register is 0x2102

>>> show ip interface brief
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.0.0.1        YES NVRAM  up                    up
GigabitEthernet0/1     192.168.1.1     YES NVRAM  up                    up
Loopback0              1.1.1.1         YES NVRAM  up                    up
GigabitEthernet0/2     unassigned      YES NVRAM  administratively down down

>>> show ip route
Codes: C - connected, S - static
C    10.0.0.0/24 is directly connected, GigabitEthernet0/0
C    192.168.1.0/24 is directly connecte

### 1.3 SFTP — Automated Config Backup

In [4]:
FAKE_CFG = """\
! Running configuration — RTR-CORE-01
version 15.2
hostname RTR-CORE-01
!
interface GigabitEthernet0/0
 ip address 10.0.0.1 255.255.255.0
 no shutdown
!
ip route 0.0.0.0 0.0.0.0 10.0.0.254
!
end
"""

class MockSFTP:
    """Simulates paramiko.SFTPClient."""
    def get(self, remote_path, local_path):
        Path(local_path).write_text(FAKE_CFG)
        print(f"  📥 SFTP: {remote_path} → {local_path}")
    def close(self): pass


def backup_config_sftp(host, username, password, remote_path, backup_dir):
    """
    Pull a device config via SFTP and save locally.
    🔌 REAL DEVICE: replace MockSFTP() with
       paramiko.SFTPClient.from_transport(transport)
    """
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    local = Path(backup_dir) / f"{host.replace('.','_')}_{ts}.cfg"
    sftp = MockSFTP()
    try:
        sftp.get(remote_path, str(local))
    finally:
        sftp.close()
    print(f"  ✅ Saved: {local.name}  ({local.stat().st_size} bytes)")
    return str(local)


bk = backup_config_sftp("192.168.1.1", "admin", "cisco123",
                         "/flash/running-config", LAB_DIR)
print("\n--- Backed-up Config ---")
print(open(bk).read())

  📥 SFTP: /flash/running-config → /content/lab2/192_168_1_1_20260507_213320.cfg
  ✅ Saved: 192_168_1_1_20260507_213320.cfg  (197 bytes)

--- Backed-up Config ---
! Running configuration — RTR-CORE-01
version 15.2
hostname RTR-CORE-01
!
interface GigabitEthernet0/0
 ip address 10.0.0.1 255.255.255.0
 no shutdown
!
ip route 0.0.0.0 0.0.0.0 10.0.0.254
!
end



### ✏️ Exercise 1 — Bulk SSH Command Runner

Write `bulk_ssh_runner(device_list, commands)` that:
1. Calls `ssh_run_commands()` for each device dict `{host, username, password}`.
2. Saves each device's output to `LAB_DIR/<host>_output.txt`.
3. Returns `{host: {"commands_run": N, "file": path}}`.
4. Prints a summary table.

In [9]:
DEMO_DEVICES = [
    {"host": "10.0.0.1",    "username": "admin", "password": "cisco123"},
    {"host": "10.0.0.2",    "username": "admin", "password": "cisco123"},
    {"host": "192.168.1.1", "username": "admin", "password": "cisco123"}
]

def bulk_ssh_runner(device_list, commands):
    results = {}

    print(f"{'Host':<18} {'Commands Run':>14} {'File'}")
    print("-" * 65)

    for device in device_list:
        host = device["host"]
        try:
            output = ssh_run_commands(
                host=host,
                username=device["username"],
                password=device["password"],
                commands=commands
            )
            # Save output to file
            out_file = LAB_DIR / f"{host}_output.txt"
            with open(out_file, "w") as f:
                for cmd, out in output.items():
                    f.write(f">>> {cmd}\n{out}\n\n")

            results[host] = {
                "commands_run": len(output),
                "file": str(out_file)
            }
            print(f"{host:<18} {len(output):>14} {out_file.name}")

        except Exception as e:
            results[host] = {"commands_run": 0, "file": None, "error": str(e)}
            print(f"{host:<18} {'ERROR':>14} {e}")

    return results

bulk_ssh_runner(DEMO_DEVICES, ["show version", "show ip interface brief"])


Host                 Commands Run File
-----------------------------------------------------------------
  🔐 [Paramiko] Connected to 10.0.0.1:22 as 'admin'
  ✅ Executed: 'show version'
  ✅ Executed: 'show ip interface brief'
  🔒 [Paramiko] Connection to 10.0.0.1 closed
10.0.0.1                        2 10.0.0.1_output.txt
  🔐 [Paramiko] Connected to 10.0.0.2:22 as 'admin'
  ✅ Executed: 'show version'
  ✅ Executed: 'show ip interface brief'
  🔒 [Paramiko] Connection to 10.0.0.2 closed
10.0.0.2                        2 10.0.0.2_output.txt
  🔐 [Paramiko] Connected to 192.168.1.1:22 as 'admin'
  ✅ Executed: 'show version'
  ✅ Executed: 'show ip interface brief'
  🔒 [Paramiko] Connection to 192.168.1.1 closed
192.168.1.1                     2 192.168.1.1_output.txt


{'10.0.0.1': {'commands_run': 2, 'file': '/content/lab2/10.0.0.1_output.txt'},
 '10.0.0.2': {'commands_run': 2, 'file': '/content/lab2/10.0.0.2_output.txt'},
 '192.168.1.1': {'commands_run': 2,
  'file': '/content/lab2/192.168.1.1_output.txt'}}

---
## Part 2 — Netmiko: Multi-Vendor SSH Automation

**Netmiko** wraps Paramiko with device-specific intelligence — prompt handling, pagination, `enable` mode, and config mode — across 50+ vendor platforms.

| Feature | Paramiko | Netmiko |
|---|---|---|
| Level | Low-level SSH | High-level, vendor-aware |
| Prompt handling | Manual | Automatic |
| Enable / config mode | Manual | Built-in |
| Vendor support | Any SSH | 50+ network vendors |

### 2.1 Simulated Netmiko Session

In [10]:
NETMIKO_COMMANDS = {
    "show version": """\
Cisco IOS Software, Version 15.4(3)M2
Router uptime is 12 days, 3 hours, 45 minutes
cisco ISR4321/K9 processor — 2 Gigabit Ethernet interfaces""",

    "show ip interface brief": """\
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0/0   10.0.0.1        YES NVRAM  up                    up
GigabitEthernet0/0/1   172.16.0.1      YES NVRAM  up                    up
GigabitEthernet0/0/2   unassigned      YES NVRAM  administratively down down""",

    "show ip bgp summary": """\
BGP router identifier 1.1.1.1, local AS number 65001
Neighbor        V    AS MsgRcvd MsgSent   Up/Down  State/PfxRcd
10.0.0.2        4 65002     120     115   01:22:33        15
172.16.0.2      4 65003      98      95   00:55:12         8""",

    "show cdp neighbors": """\
Device ID        Local Intrfce     Holdtme  Platform  Port ID
SW-ACCESS-01     Gig 0/0/0         120      WS-C2960  Gig 0/1
RTR-BRANCH-01    Gig 0/0/1         150      ISR4221   Gig 0/0"""
}


class MockNetmikoConnection:
    """Simulates a netmiko ConnectHandler session."""
    def __init__(self, device_type, host, username, password, secret="", **kw):
        self.device_type = device_type
        self.host = host
        self._config_applied = []
        time.sleep(0.2)
        print(f"  ✅ [Netmiko] Connected to {host} ({device_type})")

    def send_command(self, command, **kw):
        return NETMIKO_COMMANDS.get(command.strip(),
               f"% Unrecognised: {command}")

    def send_config_set(self, config_commands):
        if isinstance(config_commands, str):
            config_commands = config_commands.splitlines()
        self._config_applied.extend(config_commands)
        return "\n".join(f"{self.host}(config)#{c}" for c in config_commands)

    def save_config(self):
        return "Building configuration...\n[OK]"

    def find_prompt(self):
        return f"{self.host}#"

    def disconnect(self):
        print(f"  🔒 [Netmiko] Disconnected from {self.host}")

    def __enter__(self): return self
    def __exit__(self, *a): self.disconnect()


NETMIKO_DEVICE = {
    "device_type": "cisco_ios", "host": "192.168.1.1",
    "username": "admin", "password": "cisco123", "secret": ""
}
print("✅ Mock Netmiko environment ready.")

✅ Mock Netmiko environment ready.


### 2.2 Reading Device State

In [11]:
# 🔌 REAL DEVICE: from netmiko import ConnectHandler
#                 with ConnectHandler(**NETMIKO_DEVICE) as net_connect:

with MockNetmikoConnection(**NETMIKO_DEVICE) as net_connect:
    print(f"\n  Prompt: {net_connect.find_prompt()}\n")
    for cmd in ["show version", "show ip interface brief", "show cdp neighbors"]:
        out = net_connect.send_command(cmd)
        print(f"{'='*50}\n>>> {cmd}\n{out}\n")

  ✅ [Netmiko] Connected to 192.168.1.1 (cisco_ios)

  Prompt: 192.168.1.1#

>>> show version
Cisco IOS Software, Version 15.4(3)M2
Router uptime is 12 days, 3 hours, 45 minutes
cisco ISR4321/K9 processor — 2 Gigabit Ethernet interfaces

>>> show ip interface brief
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0/0   10.0.0.1        YES NVRAM  up                    up
GigabitEthernet0/0/1   172.16.0.1      YES NVRAM  up                    up
GigabitEthernet0/0/2   unassigned      YES NVRAM  administratively down down

>>> show cdp neighbors
Device ID        Local Intrfce     Holdtme  Platform  Port ID
SW-ACCESS-01     Gig 0/0/0         120      WS-C2960  Gig 0/1
RTR-BRANCH-01    Gig 0/0/1         150      ISR4221   Gig 0/0

  🔒 [Netmiko] Disconnected from 192.168.1.1


### 2.3 Pushing Configurations with `send_config_set()`

In [12]:
def push_ntp_config(device_params, ntp_servers):
    """Push NTP and timezone configuration to a device."""
    config = ["! --- NTP Configuration ---"]
    for s in ntp_servers:
        config.append(f"ntp server {s}")
    config += [
        "ntp update-calendar",
        "clock timezone EAT 3 0",
        "service timestamps log datetime msec localtime show-timezone"
    ]

    print(f"\n📤 Pushing NTP config to {device_params['host']}")
    for c in config: print(f"   {c}")

    with MockNetmikoConnection(**device_params) as conn:
        out  = conn.send_config_set(config)
        save = conn.save_config()

    print(f"\n  Device output:\n{out}")
    print(f"  Save: {save}")
    return True


push_ntp_config(NETMIKO_DEVICE, ["196.15.0.1", "pool.ntp.org"])


📤 Pushing NTP config to 192.168.1.1
   ! --- NTP Configuration ---
   ntp server 196.15.0.1
   ntp server pool.ntp.org
   ntp update-calendar
   clock timezone EAT 3 0
   service timestamps log datetime msec localtime show-timezone
  ✅ [Netmiko] Connected to 192.168.1.1 (cisco_ios)
  🔒 [Netmiko] Disconnected from 192.168.1.1

  Device output:
192.168.1.1(config)#! --- NTP Configuration ---
192.168.1.1(config)#ntp server 196.15.0.1
192.168.1.1(config)#ntp server pool.ntp.org
192.168.1.1(config)#ntp update-calendar
192.168.1.1(config)#clock timezone EAT 3 0
192.168.1.1(config)#service timestamps log datetime msec localtime show-timezone
  Save: Building configuration...
[OK]


True

### ✏️ Exercise 2 — Interface Description Updater

Write `update_interface_descriptions(device_params, description_map)` where `description_map` is `{interface: description}`. Build the correct IOS config commands, push via `send_config_set()`, save, and confirm each update.

Expected IOS syntax:
```
interface GigabitEthernet0/0/0
 description WAN-Uplink
```

In [14]:
# ✏️ Your code here
DESC_MAP = {
    "GigabitEthernet0/0/0": "WAN-Uplink-ISP1",
    "GigabitEthernet0/0/1": "LAN-Core-Switch",
    "GigabitEthernet0/0/2": "Reserved-Backup"
}

def update_interface_descriptions(device_params, description_map):
    config_commands = []

    for interface, description in description_map.items():
        config_commands.append(f"interface {interface}")
        config_commands.append(f" description {description}")

    print(f"\n📤 Pushing interface descriptions to {device_params['host']}")
    print("   Config lines to push:")
    for line in config_commands:
        print(f"   {line}")

    with MockNetmikoConnection(**device_params) as conn:
        output = conn.send_config_set(config_commands)
        save = conn.save_config()

    print(f"\n  Device output:\n{output}")
    print(f"  Save: {save}")

    # Confirm each update
    print("\n  ✅ Confirmation:")
    for iface, desc in description_map.items():
        print(f"     {iface} → '{desc}'")

    return True

update_interface_descriptions(NETMIKO_DEVICE, DESC_MAP)



📤 Pushing interface descriptions to 192.168.1.1
   Config lines to push:
   interface GigabitEthernet0/0/0
    description WAN-Uplink-ISP1
   interface GigabitEthernet0/0/1
    description LAN-Core-Switch
   interface GigabitEthernet0/0/2
    description Reserved-Backup
  ✅ [Netmiko] Connected to 192.168.1.1 (cisco_ios)
  🔒 [Netmiko] Disconnected from 192.168.1.1

  Device output:
192.168.1.1(config)#interface GigabitEthernet0/0/0
192.168.1.1(config)# description WAN-Uplink-ISP1
192.168.1.1(config)#interface GigabitEthernet0/0/1
192.168.1.1(config)# description LAN-Core-Switch
192.168.1.1(config)#interface GigabitEthernet0/0/2
192.168.1.1(config)# description Reserved-Backup
  Save: Building configuration...
[OK]

  ✅ Confirmation:
     GigabitEthernet0/0/0 → 'WAN-Uplink-ISP1'
     GigabitEthernet0/0/1 → 'LAN-Core-Switch'
     GigabitEthernet0/0/2 → 'Reserved-Backup'


True

---
## Part 3 — NAPALM: Vendor-Agnostic Network State Management

**NAPALM** provides a **unified API** across vendors — the same `get_interfaces()` call works on Cisco IOS, Junos, EOS, and NX-OS, returning identical structured dictionaries.

| Feature | Netmiko | NAPALM |
|---|---|---|
| Output | Raw text | Structured dicts |
| Config apply | `send_config_set()` | `load + commit_config()` |
| Diff before apply | Manual | `compare_config()` built-in |
| Rollback | Manual | `rollback()` built-in |
| Getters | Write yourself | 20+ pre-built |

### 3.1 Simulated NAPALM Driver

In [15]:
class MockNAPALMDriver:
    """Simulates a NAPALM ios driver."""
    def __init__(self, hostname, username, password, optional_args=None):
        self.hostname = hostname
        self._candidate = ""

    def open(self):
        time.sleep(0.2)
        print(f"  ✅ [NAPALM] Opened → {self.hostname}")

    def close(self):
        print(f"  🔒 [NAPALM] Closed → {self.hostname}")

    def get_facts(self):
        return {
            "hostname": "RTR-CORE-01", "vendor": "Cisco",
            "model": "ISR4321", "os_version": "IOS 15.4(3)M2",
            "serial_number": "FTX1901AL1J", "uptime": 1050000,
            "interface_list": ["GigabitEthernet0/0/0",
                               "GigabitEthernet0/0/1",
                               "GigabitEthernet0/0/2", "Loopback0"]
        }

    def get_interfaces(self):
        return {
            "GigabitEthernet0/0/0": {
                "is_up": True,  "is_enabled": True,
                "description": "WAN-Uplink", "speed": 1000, "mtu": 1500,
                "mac_address": "AA:BB:CC:00:01:00"
            },
            "GigabitEthernet0/0/1": {
                "is_up": True,  "is_enabled": True,
                "description": "LAN-Core", "speed": 1000, "mtu": 1500,
                "mac_address": "AA:BB:CC:00:01:01"
            },
            "GigabitEthernet0/0/2": {
                "is_up": False, "is_enabled": False,
                "description": "", "speed": 1000, "mtu": 1500,
                "mac_address": "AA:BB:CC:00:01:02"
            }
        }

    def get_interfaces_ip(self):
        return {
            "GigabitEthernet0/0/0": {"ipv4": {"10.0.0.1": {"prefix_length": 24}}},
            "GigabitEthernet0/0/1": {"ipv4": {"172.16.0.1": {"prefix_length": 24}}}
        }

    def get_bgp_neighbors(self):
        return {"global": {"router_id": "1.1.1.1", "peers": {
            "10.0.0.2":   {"remote_as": 65002, "is_up": True,
                           "address_family": {"ipv4": {"received_prefixes": 15, "sent_prefixes": 8}}},
            "172.16.0.2": {"remote_as": 65003, "is_up": True,
                           "address_family": {"ipv4": {"received_prefixes": 8,  "sent_prefixes": 3}}}
        }}}

    def load_merge_candidate(self, config=None, filename=None):
        if filename: config = Path(filename).read_text()
        self._candidate = config
        print(f"  📝 [NAPALM] Candidate loaded ({len(config)} chars)")

    def compare_config(self):
        return "\n".join(
            f"+ {l}" for l in self._candidate.splitlines()
            if l.strip() and not l.startswith("!")
        )

    def commit_config(self):
        self._candidate = ""
        print("  ✅ [NAPALM] Config committed")

    def discard_config(self):
        self._candidate = ""
        print("  ♻️  [NAPALM] Candidate discarded")

    def rollback(self):
        print("  🔄 [NAPALM] Rolled back")

    def __enter__(self):
        self.open(); return self

    def __exit__(self, *a): self.close()


print("✅ Mock NAPALM driver ready.")

✅ Mock NAPALM driver ready.


### 3.2 Using NAPALM Getters

In [16]:
# 🔌 REAL DEVICE:
#   from napalm import get_network_driver
#   driver = get_network_driver('ios')
#   conn   = driver('192.168.1.1', 'admin', 'cisco123')

with MockNAPALMDriver("192.168.1.1", "admin", "cisco123") as conn:

    # get_facts
    facts = conn.get_facts()
    print("\n📋 Device Facts")
    for k, v in facts.items():
        if k != "interface_list":
            print(f"  {k:<18}: {v}")

    # get_interfaces
    intfs = conn.get_interfaces()
    print("\n🔌 Interfaces")
    print(f"{'Interface':<25} {'Up':>4} {'Enabled':>8} {'Desc'}")
    print("-" * 60)
    for name, d in intfs.items():
        icon = "🟢" if d["is_up"] else "🔴"
        print(f"{name:<25} {icon:>4} {str(d['is_enabled']):>8}  {d['description']}")

    # get_bgp_neighbors
    bgp   = conn.get_bgp_neighbors()
    peers = bgp["global"]["peers"]
    print(f"\n📡 BGP  (Router-ID: {bgp['global']['router_id']})")
    print(f"{'Peer':<16} {'AS':>6} {'Up':>4} {'Rcvd':>8} {'Sent':>8}")
    print("-" * 46)
    for peer, pd in peers.items():
        af = pd["address_family"]["ipv4"]
        icon = "🟢" if pd["is_up"] else "🔴"
        print(f"{peer:<16} {pd['remote_as']:>6} {icon:>4} {af['received_prefixes']:>8} {af['sent_prefixes']:>8}")

  ✅ [NAPALM] Opened → 192.168.1.1

📋 Device Facts
  hostname          : RTR-CORE-01
  vendor            : Cisco
  model             : ISR4321
  os_version        : IOS 15.4(3)M2
  serial_number     : FTX1901AL1J
  uptime            : 1050000

🔌 Interfaces
Interface                   Up  Enabled Desc
------------------------------------------------------------
GigabitEthernet0/0/0         🟢     True  WAN-Uplink
GigabitEthernet0/0/1         🟢     True  LAN-Core
GigabitEthernet0/0/2         🔴    False  

📡 BGP  (Router-ID: 1.1.1.1)
Peer                 AS   Up     Rcvd     Sent
----------------------------------------------
10.0.0.2          65002    🟢       15        8
172.16.0.2        65003    🟢        8        3
  🔒 [NAPALM] Closed → 192.168.1.1


### 3.3 Safe Config Push: Load → Diff → Commit / Discard

In [17]:
NTP_CONFIG = """\
! NTP update — automated push
ntp server 196.15.0.1
ntp server pool.ntp.org
logging buffered 10000
ip domain-name strathmore.local
"""

def safe_config_push(driver, config_text, dry_run=False):
    """Load config, show diff, then commit or discard."""
    driver.load_merge_candidate(config=config_text)
    diff = driver.compare_config()
    print("\n📊 Config Diff (+ = lines to add):")
    print("-" * 45)
    print(diff or "  (no changes)")
    print("-" * 45)

    if dry_run:
        print("\n⚠️  DRY RUN — discarding")
        driver.discard_config()
    else:
        print("\n🚀 Committing ...")
        driver.commit_config()


with MockNAPALMDriver("192.168.1.1", "admin", "cisco123") as c:
    print("--- DRY RUN ---")
    safe_config_push(c, NTP_CONFIG, dry_run=True)

print()
with MockNAPALMDriver("192.168.1.1", "admin", "cisco123") as c:
    print("--- LIVE PUSH ---")
    safe_config_push(c, NTP_CONFIG, dry_run=False)

  ✅ [NAPALM] Opened → 192.168.1.1
--- DRY RUN ---
  📝 [NAPALM] Candidate loaded (131 chars)

📊 Config Diff (+ = lines to add):
---------------------------------------------
+ ntp server 196.15.0.1
+ ntp server pool.ntp.org
+ logging buffered 10000
+ ip domain-name strathmore.local
---------------------------------------------

⚠️  DRY RUN — discarding
  ♻️  [NAPALM] Candidate discarded
  🔒 [NAPALM] Closed → 192.168.1.1

  ✅ [NAPALM] Opened → 192.168.1.1
--- LIVE PUSH ---
  📝 [NAPALM] Candidate loaded (131 chars)

📊 Config Diff (+ = lines to add):
---------------------------------------------
+ ntp server 196.15.0.1
+ ntp server pool.ntp.org
+ logging buffered 10000
+ ip domain-name strathmore.local
---------------------------------------------

🚀 Committing ...
  ✅ [NAPALM] Config committed
  🔒 [NAPALM] Closed → 192.168.1.1


### ✏️ Exercise 3 — NAPALM Compliance Checker

Write `compliance_check(driver)` that:
1. Flags enabled interfaces with an **empty description** (policy violation).
2. Flags BGP peers where `is_up=False`.
3. Prints a PASS / FAIL report.
4. Returns `True` if all checks pass.

In [18]:
# ✏️ Your code here

def compliance_check(driver):
    all_pass = True
    issues = []

    # Check 1: Enabled interfaces must have a description
    interfaces = driver.get_interfaces()
    print("\n🔍 Interface Description Check:")
    for name, data in interfaces.items():
        if data["is_enabled"]:
            if not data["description"].strip():
                issues.append(f"FAIL: {name} is enabled but has no description")
                print(f"  ❌ {name} — enabled, missing description")
                all_pass = False
            else:
                print(f"  ✅ {name} — '{data['description']}'")

    # Check 2: BGP peers must be up
    bgp = driver.get_bgp_neighbors()
    peers = bgp.get("global", {}).get("peers", {})
    print("\n🔍 BGP Peer Status Check:")
    for peer_ip, peer_data in peers.items():
        if not peer_data["is_up"]:
            issues.append(f"FAIL: BGP peer {peer_ip} is DOWN")
            print(f"  ❌ Peer {peer_ip} (AS {peer_data['remote_as']}) — DOWN")
            all_pass = False
        else:
            print(f"  ✅ Peer {peer_ip} (AS {peer_data['remote_as']}) — UP")

    # Print report
    print("\n" + "=" * 45)
    print("  COMPLIANCE REPORT")
    print("=" * 45)
    if issues:
        for issue in issues:
            print(f"  {issue}")
    else:
        print("  All checks passed.")
    print("=" * 45)

    return all_pass

with MockNAPALMDriver("192.168.1.1", "admin", "cisco123") as c:
    ok = compliance_check(c)
    print(f"\nOverall: {'✅ COMPLIANT' if ok else '❌ NON-COMPLIANT'}")
def compliance_check(driver):
    # TODO: implement
    pass

with MockNAPALMDriver("192.168.1.1", "admin", "cisco123") as c:
    ok = compliance_check(c)
    print(f"\nOverall: {'✅ COMPLIANT' if ok else '❌ NON-COMPLIANT'}")

  ✅ [NAPALM] Opened → 192.168.1.1

🔍 Interface Description Check:
  ✅ GigabitEthernet0/0/0 — 'WAN-Uplink'
  ✅ GigabitEthernet0/0/1 — 'LAN-Core'

🔍 BGP Peer Status Check:
  ✅ Peer 10.0.0.2 (AS 65002) — UP
  ✅ Peer 172.16.0.2 (AS 65003) — UP

  COMPLIANCE REPORT
  All checks passed.

Overall: ✅ COMPLIANT
  🔒 [NAPALM] Closed → 192.168.1.1
  ✅ [NAPALM] Opened → 192.168.1.1

Overall: ❌ NON-COMPLIANT
  🔒 [NAPALM] Closed → 192.168.1.1


---
## Part 4 — Nornir: Parallel Network Automation Framework

**Nornir** manages an **inventory** of devices and runs **tasks** against them — in parallel by default — using pure Python.

| Concept | Meaning |
|---|---|
| **Inventory** | YAML files — hosts, groups, defaults |
| **Task** | Python function that runs on one host |
| **Result** | Output + success/fail metadata |
| **Runner** | Thread pool controls parallelism |
| **Filter** | Select subset of hosts by attribute |

### 4.1 Building the Inventory

In [19]:
NORNIR_DIR = LAB_DIR / "nornir"
NORNIR_DIR.mkdir(exist_ok=True)

(NORNIR_DIR / "hosts.yaml").write_text("""\
RTR-CORE-01:
  hostname: 192.168.1.1
  groups: [routers, site-nairobi]
  data: {role: core-router, location: Server Room A}

RTR-BRANCH-01:
  hostname: 10.10.2.1
  groups: [routers, site-mombasa]
  data: {role: branch-router, location: Mombasa Office}

SW-ACCESS-01:
  hostname: 192.168.1.10
  groups: [switches, site-nairobi]
  data: {role: access-switch, location: Floor 2 MDF}

SW-ACCESS-02:
  hostname: 192.168.1.11
  groups: [switches, site-nairobi]
  data: {role: access-switch, location: Floor 3 MDF}

FW-EDGE-01:
  hostname: 10.0.0.254
  groups: [firewalls, site-nairobi]
  data: {role: edge-firewall, location: DMZ}
""")

(NORNIR_DIR / "groups.yaml").write_text("""\
routers:
  platform: ios
  data: {device_type: router}
switches:
  platform: ios
  data: {device_type: switch}
firewalls:
  platform: asa
  data: {device_type: firewall}
site-nairobi:
  data: {site: Nairobi, country: Kenya}
site-mombasa:
  data: {site: Mombasa, country: Kenya}
""")

(NORNIR_DIR / "defaults.yaml").write_text("""\
username: admin
password: cisco123
data:
  domain: strathmore.local
  ntp_server: 196.15.0.1
""")

print("✅ Nornir inventory files written:")
for f in sorted(NORNIR_DIR.iterdir()):
    print(f"   {f.name}")

✅ Nornir inventory files written:
   defaults.yaml
   groups.yaml
   hosts.yaml


### 4.2 Initialising Nornir & Filtering Hosts

In [20]:
from nornir import InitNornir
from nornir.core.filter import F

# 🔌 REAL DEVICE: this exact code works unchanged — just update
#    hostnames/credentials in hosts.yaml and defaults.yaml
nr = InitNornir(
    runner={"plugin": "threaded", "options": {"num_workers": 5}},
    inventory={
        "plugin": "SimpleInventory",
        "options": {
            "host_file":     str(NORNIR_DIR / "hosts.yaml"),
            "group_file":    str(NORNIR_DIR / "groups.yaml"),
            "defaults_file": str(NORNIR_DIR / "defaults.yaml")
        }
    }
)

print(f"✅ Nornir ready")
print(f"   Hosts : {len(nr.inventory.hosts)}")
print(f"   Groups: {len(nr.inventory.groups)}")

nbi_routers = nr.filter(F(groups__contains="site-nairobi") & F(groups__contains="routers"))
print(f"\n🔍 Nairobi routers : {list(nbi_routers.inventory.hosts.keys())}")

all_switches = nr.filter(F(groups__contains="switches"))
print(f"🔍 All switches    : {list(all_switches.inventory.hosts.keys())}")

✅ Nornir ready
   Hosts : 5
   Groups: 5

🔍 Nairobi routers : ['RTR-CORE-01']
🔍 All switches    : ['SW-ACCESS-01', 'SW-ACCESS-02']


/usr/local/lib/python3.12/dist-packages/nornir/core/configuration.py:160: ConflictingConfigurationWarning: Native Python logging configuration has been detected, but Nornir logging is enabled too. This can lead to unexpected logging results. Please set logging.enabled config to False to disable automatic Nornir logging configuration. Refer to https://nornir.readthedocs.io/en/stable/configuration/index.html#logging
  warnings.warn(msg, ConflictingConfigurationWarning)


### 4.3 Writing and Running Tasks

In [21]:
from nornir.core.task import Task, Result

def collect_inventory_task(task: Task) -> Result:
    """
    Simulate collecting device info.
    🔌 REAL DEVICE: replace mock dict with Netmiko/NAPALM calls.
    """
    return Result(host=task.host, result={
        "hostname":    task.host.name,
        "ip":          str(task.host.hostname),
        "platform":    task.host.platform or "unknown",
        "role":        task.host.data.get("role", "unknown"),
        "site":        task.host.data.get("site", "unknown"),
        "status":      "reachable",
        "checked_at":  datetime.datetime.now().isoformat()
    })


print("🚀 Running collect_inventory_task across all hosts ...\n")
results = nr.run(task=collect_inventory_task)

print(f"\n{'Host':<22} {'Role':<18} {'Site':<12} {'Platform':<8} {'Status'}")
print("-" * 68)
for hostname, multi in results.items():
    f = multi[0].result
    icon = "✅" if f["status"] == "reachable" else "❌"
    print(f"{hostname:<22} {f['role']:<18} {f['site']:<12} {f['platform']:<8} {icon}")

🚀 Running collect_inventory_task across all hosts ...


Host                   Role               Site         Platform Status
--------------------------------------------------------------------
RTR-CORE-01            core-router        unknown      ios      ✅
RTR-BRANCH-01          branch-router      unknown      ios      ✅
SW-ACCESS-01           access-switch      unknown      ios      ✅
SW-ACCESS-02           access-switch      unknown      ios      ✅
FW-EDGE-01             edge-firewall      unknown      asa      ✅


### 4.4 Multi-Step Tasks with Sub-Tasks

In [22]:
def check_ntp_compliance(task: Task) -> Result:
    # Routers compliant, switches not — simulated
    compliant = "routers" in [str(g) for g in task.host.groups]
    return Result(host=task.host,
                  result={"ntp_compliant": compliant})

def check_ssh_enabled(task: Task) -> Result:
    return Result(host=task.host, result={"ssh_v2_enabled": True})

def full_compliance_audit(task: Task) -> Result:
    ntp = task.run(task=check_ntp_compliance)[0].result["ntp_compliant"]
    ssh = task.run(task=check_ssh_enabled)[0].result["ssh_v2_enabled"]
    return Result(host=task.host,
                  result={"ntp": ntp, "ssh": ssh, "overall": ntp and ssh})


print("🔍 Compliance audit across all devices ...\n")
audit = nr.run(task=full_compliance_audit)

print(f"\n{'Host':<22} {'NTP':>6} {'SSH':>6} {'Overall':>9}")
print("-" * 48)
for h, multi in audit.items():
    r = multi[0].result
    print(f"{h:<22} {'✅' if r['ntp'] else '❌':>6} {'✅' if r['ssh'] else '❌':>6} {'✅' if r['overall'] else '❌':>9}")

🔍 Compliance audit across all devices ...


Host                      NTP    SSH   Overall
------------------------------------------------
RTR-CORE-01                 ✅      ✅         ✅
RTR-BRANCH-01               ✅      ✅         ✅
SW-ACCESS-01                ❌      ✅         ❌
SW-ACCESS-02                ❌      ✅         ❌
FW-EDGE-01                  ❌      ✅         ❌


### ✏️ Exercise 4 — Nornir Config Backup Task

Write `backup_running_config(task)` that:
1. Returns a mock running-config string.
2. Saves it to `LAB_DIR/backups/<hostname>_<YYYYMMDD>.cfg`.
3. Returns `Result` with `{"file": path, "size_bytes": N}`.

Run across the whole inventory and print a summary table.

In [23]:
# ✏️ Your code here

(LAB_DIR / "backups").mkdir(exist_ok=True)

def backup_running_config(task: Task) -> Result:
    # Simulated running config for this device
    mock_config = f"""\
! Running configuration — {task.host.name}
! Generated: {datetime.datetime.now().isoformat()}
!
version 15.4
hostname {task.host.name}
!
interface GigabitEthernet0/0
 ip address {task.host.hostname} 255.255.255.0
 no shutdown
!
ip route 0.0.0.0 0.0.0.0 10.0.0.254
!
end
"""
    date_str = datetime.datetime.now().strftime("%Y%m%d")
    file_path = LAB_DIR / "backups" / f"{task.host.name}_{date_str}.cfg"
    file_path.write_text(mock_config)

    return Result(host=task.host, result={
        "file": str(file_path),
        "size_bytes": file_path.stat().st_size
    })


backup_results = nr.run(task=backup_running_config)

print(f"\n{'Host':<22} {'File':<45} {'Size':>10}")
print("-" * 80)
for hostname, multi in backup_results.items():
    r = multi[0].result
    filename = Path(r["file"]).name
    print(f"{hostname:<22} {filename:<45} {r['size_bytes']:>8} B")
(LAB_DIR / "backups").mkdir(exist_ok=True)

def backup_running_config(task: Task) -> Result:
    # TODO: implement
    pass

backup_results = nr.run(task=backup_running_config)
# TODO: print summary table


Host                   File                                                Size
--------------------------------------------------------------------------------
RTR-CORE-01            RTR-CORE-01_20260507.cfg                           242 B
RTR-BRANCH-01          RTR-BRANCH-01_20260507.cfg                         244 B
SW-ACCESS-01           SW-ACCESS-01_20260507.cfg                          245 B
SW-ACCESS-02           SW-ACCESS-02_20260507.cfg                          245 B
FW-EDGE-01             FW-EDGE-01_20260507.cfg                            239 B


---
## Part 5 — Ansible: Declarative Network Automation

**Ansible** uses YAML *playbooks* to describe **desired state**. It figures out the commands needed to reach that state.

| Aspect | Nornir | Ansible |
|---|---|---|
| Language | Pure Python | YAML + Jinja2 |
| Logic | Imperative | Declarative |
| Agentless | Yes | Yes |
| Network modules | Via plugins | `cisco.ios`, `junipernetworks.junos`, etc. |

### 5.1 Creating Inventory & Playbooks

In [24]:
ANSIBLE_DIR = LAB_DIR / "ansible"
ANSIBLE_DIR.mkdir(exist_ok=True)

(ANSIBLE_DIR / "inventory.ini").write_text("""\
[routers]
localhost ansible_connection=local

[routers:vars]
ansible_python_interpreter=/usr/bin/python3
""")

(ANSIBLE_DIR / "ansible.cfg").write_text("""\
[defaults]
inventory = inventory.ini
host_key_checking = False
retry_files_enabled = False
""")

(ANSIBLE_DIR / "network_report.yml").write_text("""\
---
- name: Network Device Report
  hosts: routers
  gather_facts: true
  vars:
    report_dir: "/content/lab2/ansible/reports"
    devices:
      - name: RTR-CORE-01
        ip: 192.168.1.1
        role: core-router
        vlans: [10, 20, 99]
      - name: SW-ACCESS-01
        ip: 192.168.1.10
        role: access-switch
        vlans: [10, 20]

  tasks:

    - name: Create report directory
      ansible.builtin.file:
        path: "{{ report_dir }}"
        state: directory
        mode: '0755'

    - name: Write device inventory JSON
      ansible.builtin.copy:
        content: "{{ devices | to_nice_json }}"
        dest: "{{ report_dir }}/device_inventory.json"

    - name: Generate per-device config stubs
      ansible.builtin.copy:
        content: |
          ! Auto-generated stub for {{ item.name }}
          hostname {{ item.name }}
          !
          {% for vlan in item.vlans %}
          vlan {{ vlan }}
          {% endfor %}
          end
        dest: "{{ report_dir }}/{{ item.name }}_stub.cfg"
      loop: "{{ devices }}"

    - name: Print summary
      ansible.builtin.debug:
        msg: "Processed {{ devices | length }} devices → {{ report_dir }}"
""")

print("✅ Ansible files ready:")
for f in sorted(ANSIBLE_DIR.iterdir()):
    print(f"   {f.name}")

✅ Ansible files ready:
   ansible.cfg
   inventory.ini
   network_report.yml


### 5.2 Running the Playbook

In [25]:
import subprocess

result = subprocess.run(
    ["ansible-playbook", "-i", "inventory.ini", "network_report.yml", "-v"],
    cwd=str(ANSIBLE_DIR), capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Using /content/lab2/ansible/ansible.cfg as config file

PLAY [Network Device Report] ***************************************************

TASK [Gathering Facts] *********************************************************
ok: [localhost]

TASK [Create report directory] *************************************************
changed: [localhost] => {"changed": true, "gid": 0, "group": "root", "mode": "0755", "owner": "root", "path": "/content/lab2/ansible/reports", "size": 4096, "state": "directory", "uid": 0}

TASK [Write device inventory JSON] *********************************************
changed: [localhost] => {"changed": true, "checksum": "4b3e93436af81582a56e608cd5b381005984d705", "dest": "/content/lab2/ansible/reports/device_inventory.json", "gid": 0, "group": "root", "md5sum": "e72bd8ad6028155355e290f4a82d72ab", "mode": "0644", "owner": "root", "size": 350, "src": "/root/.ansible/tmp/ansible-tmp-1778189997.8362925-6050-275285319431487/.source.json", "state": "file", "uid": 0}

TASK [Gene

In [26]:
reports_dir = ANSIBLE_DIR / "reports"
if reports_dir.exists():
    for f in sorted(reports_dir.iterdir()):
        print(f"\n--- {f.name} ---")
        print(f.read_text())
else:
    print("⚠️  Reports directory not created — check playbook output above.")


--- RTR-CORE-01_stub.cfg ---
! Auto-generated stub for RTR-CORE-01
hostname RTR-CORE-01
!
vlan 10
vlan 20
vlan 99
end


--- SW-ACCESS-01_stub.cfg ---
! Auto-generated stub for SW-ACCESS-01
hostname SW-ACCESS-01
!
vlan 10
vlan 20
end


--- device_inventory.json ---
[
    {
        "ip": "192.168.1.1",
        "name": "RTR-CORE-01",
        "role": "core-router",
        "vlans": [
            10,
            20,
            99
        ]
    },
    {
        "ip": "192.168.1.10",
        "name": "SW-ACCESS-01",
        "role": "access-switch",
        "vlans": [
            10,
            20
        ]
    }
]


### 5.3 NTP Deployment Playbook

In [27]:
(ANSIBLE_DIR / "deploy_ntp.yml").write_text("""\
---
- name: Deploy NTP Configuration
  hosts: routers
  gather_facts: false
  vars:
    ntp_servers: [196.15.0.1, pool.ntp.org]
    output_dir: /content/lab2/ansible/configs

  tasks:
    - name: Ensure config dir exists
      ansible.builtin.file:
        path: "{{ output_dir }}"
        state: directory

    - name: Generate NTP config snippet
      ansible.builtin.copy:
        content: |
          ! NTP Configuration — deployed by Ansible
          {% for server in ntp_servers %}
          ntp server {{ server }}
          {% endfor %}
          ntp update-calendar
          clock timezone EAT 3 0
        dest: "{{ output_dir }}/ntp_config.cfg"

    - name: Deployment summary
      ansible.builtin.debug:
        msg: "NTP servers: {{ ntp_servers | join(', ') }}"
""")

r2 = subprocess.run(
    ["ansible-playbook", "-i", "inventory.ini", "deploy_ntp.yml"],
    cwd=str(ANSIBLE_DIR), capture_output=True, text=True
)
print(r2.stdout)
if r2.returncode != 0: print(r2.stderr)

cfg_file = ANSIBLE_DIR / "configs" / "ntp_config.cfg"
if cfg_file.exists():
    print("\n--- Generated NTP config ---")
    print(cfg_file.read_text())


PLAY [Deploy NTP Configuration] ************************************************

TASK [Ensure config dir exists] ************************************************
changed: [localhost]

TASK [Generate NTP config snippet] *********************************************
changed: [localhost]

TASK [Deployment summary] ******************************************************
ok: [localhost] => {
    "msg": "NTP servers: 196.15.0.1, pool.ntp.org"
}

PLAY RECAP *********************************************************************
localhost                  : ok=3    changed=2    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   



--- Generated NTP config ---
! NTP Configuration — deployed by Ansible
ntp server 196.15.0.1
ntp server pool.ntp.org
ntp update-calendar
clock timezone EAT 3 0



### ✏️ Exercise 5 — VLAN Deployment Playbook

Write and run `deploy_vlans.yml` that:
1. Defines `vlans: [{id: 10, name: STAFF}, {id: 20, name: STUDENTS}, {id: 99, name: MGMT}]`.
2. Creates `/content/lab2/ansible/vlan_configs/`.
3. Generates `vlan_config.cfg` with proper IOS `vlan <id>` + ` name <name>` blocks.
4. Uses `debug` to print the number of VLANs configured.

In [28]:
# ✏️ Your code here

vlan_playbook = """\
---
- name: Deploy VLAN Configuration
  hosts: routers
  gather_facts: false
  vars:
    vlans:
      - {id: 10, name: STAFF}
      - {id: 20, name: STUDENTS}
      - {id: 99, name: MGMT}
    output_dir: /content/lab2/ansible/vlan_configs

  tasks:

    - name: Create vlan_configs directory
      ansible.builtin.file:
        path: "{{ output_dir }}"
        state: directory
        mode: '0755'

    - name: Generate IOS VLAN config
      ansible.builtin.copy:
        content: |
          ! VLAN Configuration — deployed by Ansible
          {% for vlan in vlans %}
          vlan {{ vlan.id }}
           name {{ vlan.name }}
          !
          {% endfor %}
        dest: "{{ output_dir }}/vlan_config.cfg"

    - name: Print VLAN summary
      ansible.builtin.debug:
        msg: "Configured {{ vlans | length }} VLANs: {{ vlans | map(attribute='name') | join(', ') }}"
"""

(ANSIBLE_DIR / "deploy_vlans.yml").write_text(vlan_playbook)

r3 = subprocess.run(
    ["ansible-playbook", "-i", "inventory.ini", "deploy_vlans.yml"],
    cwd=str(ANSIBLE_DIR), capture_output=True, text=True
)
print(r3.stdout)
if r3.returncode != 0:
    print("STDERR:", r3.stderr)

cfg = ANSIBLE_DIR / "vlan_configs" / "vlan_config.cfg"
if cfg.exists():
    print("\n--- Generated VLAN config ---")
    print(cfg.read_text())
vlan_playbook = """\
---
# TODO: write deploy_vlans.yml here
"""
(ANSIBLE_DIR / "deploy_vlans.yml").write_text(vlan_playbook)
# TODO: run with subprocess and print output


PLAY [Deploy VLAN Configuration] ***********************************************

TASK [Create vlan_configs directory] *******************************************
changed: [localhost]

TASK [Generate IOS VLAN config] ************************************************
changed: [localhost]

TASK [Print VLAN summary] ******************************************************
ok: [localhost] => {
    "msg": "Configured 3 VLANs: STAFF, STUDENTS, MGMT"
}

PLAY RECAP *********************************************************************
localhost                  : ok=3    changed=2    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   



--- Generated VLAN config ---
! VLAN Configuration — deployed by Ansible
vlan 10
 name STAFF
!
vlan 20
 name STUDENTS
!
vlan 99
 name MGMT
!



40

---
## Part 6 — REST APIs: Network Intelligence

Modern devices expose HTTP/REST management interfaces. Public APIs provide real-time network intelligence data.

| API | Provides | Auth |
|---|---|---|
| `ip-api.com` | IP geolocation + ISP | None |
| `api.bgpview.io` | BGP ASN & prefix data | None |
| `rdap.arin.net` | WHOIS / IP registry | None |
| `dns.google/resolve` | DNS-over-HTTPS | None |

### 6.1 IP Geolocation

In [29]:
def get_ip_intelligence(ip):
    try:
        r = requests.get(f"http://ip-api.com/json/{ip}", timeout=5)
        d = r.json()
        if d.get("status") == "success":
            return {"ip": ip, "country": d.get("country"), "city": d.get("city"),
                    "isp": d.get("isp"), "org": d.get("org"), "asn": d.get("as"),
                    "lat": d.get("lat"), "lon": d.get("lon"), "timezone": d.get("timezone")}
    except Exception as e:
        return {"ip": ip, "error": str(e)}
    return {"ip": ip, "error": "status failed"}


print(f"{'IP':<18} {'Country':<14} {'City':<15} {'ASN'}")
print("-" * 72)
for ip in ["8.8.8.8", "1.1.1.1", "196.207.127.1"]:
    d = get_ip_intelligence(ip)
    print(f"{d.get('ip','?'):<18} {str(d.get('country','?')):<14} "
          f"{str(d.get('city','?')):<15} {d.get('asn','?')}")
    time.sleep(0.5)

IP                 Country        City            ASN
------------------------------------------------------------------------
8.8.8.8            United States  Ashburn         AS15169 Google LLC
1.1.1.1            Australia      South Brisbane  AS13335 Cloudflare, Inc.
196.207.127.1      Mauritius      Quatre Bornes   


### 6.2 BGP ASN Lookup

In [30]:
def get_asn_info(asn):
    base = "https://api.bgpview.io"
    info = {"asn": asn}
    try:
        r = requests.get(f"{base}/asn/{asn}", timeout=8)
        if r.ok:
            d = r.json().get("data", {})
            info.update({"name": d.get("name"), "org": d.get("description_short"),
                         "country": d.get("country_code"),
                         "rir": d.get("rir_allocation", {}).get("rir_name")})
        r2 = requests.get(f"{base}/asn/{asn}/prefixes", timeout=8)
        if r2.ok:
            pfx = r2.json().get("data", {}).get("ipv4_prefixes", [])
            info["ipv4_prefixes"] = len(pfx)
            info["samples"] = [p["prefix"] for p in pfx[:3]]
        time.sleep(1)
    except Exception as e:
        info["error"] = str(e)
    return info


for asn in [33771, 15169]:   # Safaricom, Google
    i = get_asn_info(asn)
    print(f"\n  ASN {i['asn']}: {i.get('name','?')}")
    print(f"   Org     : {i.get('org','?')}")
    print(f"   Country : {i.get('country','?')}  RIR: {i.get('rir','?')}")
    print(f"   Prefixes: {i.get('ipv4_prefixes','?')} — samples: {i.get('samples',[])}")


  ASN 33771: ?
   Org     : ?
   Country : ?  RIR: ?
   Prefixes: ? — samples: []

  ASN 15169: ?
   Org     : ?
   Country : ?  RIR: ?
   Prefixes: ? — samples: []


### 6.3 RDAP / WHOIS Lookup

In [31]:
def rdap_lookup(ip):
    try:
        r = requests.get(f"https://rdap.arin.net/registry/ip/{ip}",
                         headers={"Accept": "application/json"}, timeout=8)
        if r.ok:
            d = r.json()
            org = "N/A"
            for ent in d.get("entities", []):
                for field in ent.get("vcardArray", [[], []])[1]:
                    if field[0] == "fn":
                        org = field[3]; break
            return {"ip": ip, "name": d.get("name"), "org": org,
                    "country": d.get("country"),
                    "range": f"{d.get('startAddress')} – {d.get('endAddress')}"}
        return {"ip": ip, "error": f"HTTP {r.status_code}"}
    except Exception as e:
        return {"ip": ip, "error": str(e)}


for ip in ["8.8.8.8", "1.1.1.1"]:
    r = rdap_lookup(ip)
    print(f"\n  IP {r['ip']}")
    if "error" in r:
        print(f"   Error: {r['error']}")
    else:
        for k in ["name", "org", "country", "range"]:
            print(f"   {k:<8}: {r.get(k,'N/A')}")
    time.sleep(1)


  IP 8.8.8.8
   name    : GOGL
   org     : Google LLC
   country : None
   range   : 8.8.8.0 – 8.8.8.255

  IP 1.1.1.1
   name    : APNIC-LABS
   org     : APNICRANDNET Infrastructure Contact
   country : AU
   range   : 1.1.1.0 – 1.1.1.255


### 6.4 DNS-over-HTTPS

In [32]:
def doh_query(name, rtype="A"):
    type_map = {"A":1,"NS":2,"CNAME":5,"MX":15,"AAAA":28,"TXT":16,"PTR":12}
    try:
        r = requests.get("https://dns.google/resolve",
                         params={"name": name, "type": type_map.get(rtype.upper(),1)},
                         timeout=5)
        return [{"data": a["data"], "ttl": a["TTL"]}
                for a in r.json().get("Answer", [])]
    except Exception as e:
        return [{"error": str(e)}]


for domain in ["google.com", "strathmore.edu"]:
    print(f"\n  {domain}")
    for rt in ["A", "MX", "NS"]:
        recs = doh_query(domain, rt)
        if recs and "error" not in recs[0]:
            for rec in recs[:2]:
                print(f"    [{rt}] {rec['data']}  TTL={rec['ttl']}s")
        else:
            print(f"    [{rt}] — no records")


  google.com
    [A] 64.233.180.100  TTL=300s
    [A] 64.233.180.102  TTL=300s
    [MX] 10 smtp.google.com.  TTL=9s
    [NS] ns2.google.com.  TTL=21600s
    [NS] ns1.google.com.  TTL=21600s

  strathmore.edu
    [A] 34.243.183.166  TTL=300s
    [MX] 15 aspmx4.googlemail.com.  TTL=6717s
    [MX] 5 alt1.aspmx.l.google.com.  TTL=6717s
    [NS] ns-1246.awsdns-27.org.  TTL=14400s
    [NS] ns-236.awsdns-29.com.  TTL=14400s


### ✏️ Exercise 6 — Network Threat Intelligence Dashboard

Write `threat_dashboard(ip_list)` that for each IP:
1. Calls `get_ip_intelligence()` — geolocation + ISP.
2. Calls `rdap_lookup()` — registry owner.
3. Does a reverse PTR lookup via `doh_query("<reversed>.in-addr.arpa", "PTR")`.
4. Merges into one dict per IP.
5. Saves everything to `LAB_DIR/threat_dashboard.json`.
6. Prints a formatted summary table.

In [33]:
# ✏️ Your code here

def threat_dashboard(ip_list):
    dashboard = []

    for ip in ip_list:
        print(f"\n🔍 Gathering intel for {ip} ...")

        # Step 1: Geolocation + ISP
        geo = get_ip_intelligence(ip)
        time.sleep(0.5)

        # Step 2: RDAP / Registry owner
        rdap = rdap_lookup(ip)
        time.sleep(0.5)

        # Step 3: Reverse PTR lookup
        octets = ip.split(".")
        reversed_ip = ".".join(reversed(octets)) + ".in-addr.arpa"
        ptr_records = doh_query(reversed_ip, "PTR")
        ptr = ptr_records[0].get("data", "N/A") if ptr_records else "N/A"

        # Step 4: Merge into one record
        record = {
            "ip": ip,
            "country": geo.get("country"),
            "city": geo.get("city"),
            "isp": geo.get("isp"),
            "asn": geo.get("asn"),
            "registry_name": rdap.get("name"),
            "registry_org": rdap.get("org"),
            "ip_range": rdap.get("range"),
            "ptr_record": ptr
        }
        dashboard.append(record)

    # Step 5: Save to JSON
    out_file = LAB_DIR / "threat_dashboard.json"
    out_file.write_text(json.dumps(dashboard, indent=2))
    print(f"\n✅ Saved to {out_file}")

    # Step 6: Print summary table
    print(f"\n{'IP':<16} {'Country':<12} {'ISP':<30} {'PTR'}")
    print("-" * 80)
    for r in dashboard:
        print(f"{r['ip']:<16} {str(r.get('country','?')):<12} "
              f"{str(r.get('isp','?'))[:28]:<30} {r.get('ptr_record','N/A')}")

    return dashboard

threat_dashboard(["8.8.8.8", "1.1.1.1"])
def threat_dashboard(ip_list):
    # TODO: implement
    pass

threat_dashboard(["8.8.8.8", "1.1.1.1"])


🔍 Gathering intel for 8.8.8.8 ...

🔍 Gathering intel for 1.1.1.1 ...

✅ Saved to /content/lab2/threat_dashboard.json

IP               Country      ISP                            PTR
--------------------------------------------------------------------------------
8.8.8.8          United States Google LLC                     dns.google.
1.1.1.1          Australia    Cloudflare, Inc                one.one.one.one.


---
## 🏆 Capstone Challenge — Full Automation Pipeline

Implement `run_full_pipeline(devices, nornir_instance)` combining all six tools:

| Step | Tool | Action |
|---|---|---|
| 1 | **Paramiko** | SSH → `show version` for each device |
| 2 | **Netmiko** | Push NTP config to all devices |
| 3 | **NAPALM** | Compliance check (descriptions + BGP) |
| 4 | **Nornir** | Parallel config backup |
| 5 | **Ansible** | Execute `deploy_ntp.yml` |
| 6 | **REST APIs** | Geolocate all management IPs |
| 7 | **Report** | JSON + TXT audit report in `LAB_DIR/capstone_report/` |

Each step must catch exceptions gracefully and log `PASS` / `FAIL` in the final report.

In [35]:
# 🏆 Capstone solution

def run_full_pipeline(devices, nornir_instance):
    report = {
        "generated_at": datetime.datetime.now().isoformat(),
        "steps": {}
    }

    # Step 1: Paramiko — show version
    print("\n📡 Step 1: Paramiko SSH")
    try:
        for d in devices:
            out = ssh_run_commands(d["host"], d["username"], d["password"], ["show version"])
        report["steps"]["paramiko"] = {"status": "PASS", "devices": len(devices)}
    except Exception as e:
        report["steps"]["paramiko"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['paramiko']['status']}")

    # Step 2: Netmiko — push NTP
    print("\n📡 Step 2: Netmiko NTP push")
    try:
        for d in devices:
            push_ntp_config(d, ["196.15.0.1", "pool.ntp.org"])
        report["steps"]["netmiko"] = {"status": "PASS"}
    except Exception as e:
        report["steps"]["netmiko"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['netmiko']['status']}")

    # Step 3: NAPALM — compliance check
    print("\n📡 Step 3: NAPALM compliance")
    try:
        with MockNAPALMDriver(devices[0]["host"], devices[0]["username"], devices[0]["password"]) as c:
            ok = compliance_check(c)
        report["steps"]["napalm"] = {"status": "PASS" if ok else "FAIL (non-compliant)"}
    except Exception as e:
        report["steps"]["napalm"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['napalm']['status']}")

    # Step 4: Nornir — parallel backup
    print("\n📡 Step 4: Nornir parallel backup")
    try:
        nornir_instance.run(task=backup_running_config)
        report["steps"]["nornir"] = {"status": "PASS", "devices_backed_up": len(nornir_instance.inventory.hosts)}
    except Exception as e:
        report["steps"]["nornir"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['nornir']['status']}")

    # Step 5: Ansible — deploy NTP playbook
    print("\n📡 Step 5: Ansible deploy_ntp.yml")
    try:
        r = subprocess.run(
            ["ansible-playbook", "-i", "inventory.ini", "deploy_ntp.yml"],
            cwd=str(ANSIBLE_DIR), capture_output=True, text=True
        )
        report["steps"]["ansible"] = {"status": "PASS" if r.returncode == 0 else "FAIL", "rc": r.returncode}
    except Exception as e:
        report["steps"]["ansible"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['ansible']['status']}")

    # Step 6: REST API — geolocate management IPs
    print("\n📡 Step 6: REST API geolocate")
    try:
        geo_results = [get_ip_intelligence(d["host"]) for d in devices]
        report["steps"]["rest_api"] = {"status": "PASS", "ips_checked": len(geo_results)}
    except Exception as e:
        report["steps"]["rest_api"] = {"status": "FAIL", "error": str(e)}
    print(f"  → {report['steps']['rest_api']['status']}")

    # Step 7: Save report
    report_dir = LAB_DIR / "capstone_report"
    (report_dir / "report.json").write_text(json.dumps(report, indent=2))
    txt_lines = [f"CAPSTONE REPORT — {report['generated_at']}", "=" * 50]
    for step, data in report["steps"].items():
        txt_lines.append(f"  {step.upper():<12}: {data['status']}")
    (report_dir / "report.txt").write_text("\n".join(txt_lines))

    print(f"\n✅ Report saved to {report_dir}")
    print("\n".join(txt_lines))
    return report

run_full_pipeline(PIPELINE_DEVICES, nr)

(LAB_DIR / "capstone_report").mkdir(exist_ok=True)

PIPELINE_DEVICES = [
    {"device_type": "cisco_ios", "host": "192.168.1.1",
     "username": "admin", "password": "cisco123", "secret": ""}
]

def run_full_pipeline(devices, nornir_instance):
    report = {
        "generated_at": datetime.datetime.now().isoformat(),
        "steps": {}
    }
    # TODO: implement each step
    pass

run_full_pipeline(PIPELINE_DEVICES, nr)


📡 Step 1: Paramiko SSH
  🔐 [Paramiko] Connected to 192.168.1.1:22 as 'admin'
  ✅ Executed: 'show version'
  🔒 [Paramiko] Connection to 192.168.1.1 closed
  → PASS

📡 Step 2: Netmiko NTP push

📤 Pushing NTP config to 192.168.1.1
   ! --- NTP Configuration ---
   ntp server 196.15.0.1
   ntp server pool.ntp.org
   ntp update-calendar
   clock timezone EAT 3 0
   service timestamps log datetime msec localtime show-timezone
  ✅ [Netmiko] Connected to 192.168.1.1 (cisco_ios)
  🔒 [Netmiko] Disconnected from 192.168.1.1

  Device output:
192.168.1.1(config)#! --- NTP Configuration ---
192.168.1.1(config)#ntp server 196.15.0.1
192.168.1.1(config)#ntp server pool.ntp.org
192.168.1.1(config)#ntp update-calendar
192.168.1.1(config)#clock timezone EAT 3 0
192.168.1.1(config)#service timestamps log datetime msec localtime show-timezone
  Save: Building configuration...
[OK]
  → PASS

📡 Step 3: NAPALM compliance
  ✅ [NAPALM] Opened → 192.168.1.1
  🔒 [NAPALM] Closed → 192.168.1.1
  → FAIL (non-compl

---
## 📝 Reflection Questions

**Q1.** Both Paramiko and Netmiko use SSH. Why would you choose Paramiko over Netmiko?

>
Paramiko is used when you need low-level SSH control that Netmiko doesn't expose let's say ;a custom key-based authentication flows, SFTP file transfers uploading/downloading files to/from devices, connecting to non-network devices  e.g Linux servers, IoT, or tunneling connections. Netmiko is purpose-built for network devices; Paramiko works on any SSH server.

---

**Q2.** NAPALM's `compare_config()` shows a diff before committing. Why is this critical in production?

>
It allows you see exactly what will change before anything is applied — like a git diff before a commit. In production, an accidental config change can take down services affecting thousands of users. The diff catches mistakes like a wrong interface name, missing subnet mask, accidental deletion of an ACLbefore they cause an outage. Without it, you're flying blind.

---

**Q3.** Nornir uses a thread pool. What advantage does this give? What risks does it introduce?

> The advantage is speed: running tasks on 100 devices would take roughly the same time as running on 1 device, instead of 100× longer. The risk is that concurrent writes to shared resources a single log file  can cause race conditions — two threads writing at the same time corrupting the data. You have to use thread-safe structures like queue.Queue, locks, or nornir-utils's print_result when sharing state across tasks.

---

**Q4.** Ansible is declarative; Nornir is imperative. Give one scenario where each is the better choice.

> Ansible is better when you want to enforce a desired state idempotently and the "what" matters more than the "how" — e.g., "ensure these 200 switches have NTP configured." Non-programmers can read and modify the YAML. It's also better for integration with CI/CD pipelines and Ansible Tower/AWX.
Nornir is better when you need complex conditional logic in Python — e.g., "connect to each device, parse its BGP table, and if a peer has fewer than 5 prefixes, send a Slack alert." You can't easily express that decision tree in YAML.

---

**Q5.** How does consuming a REST API differ from parsing raw SSH output e.g. `show ip interface brief`? Which is more reliable?

> REST APIs are far more reliable.They show ip interface brief output in plain text format for human eyes — spacing, column alignment, and wording can change between IOS versions, breaking your parser. A REST API returns structured JSON with consistent keys and types that don't change with software versions. REST also enables pagination, filtering, and error codes like HTTP 404 vs 500 that raw SSH output doesn't have.

---

**Q6.** You must automate config of 500 switches overnight. Which combination of tools from this lab would you use, and why?

> I would use ;
1.Ansible to push the desired configurations like VLAN, NTP, AAA .It is declarative, idempotent, and readable for auditors.
2.Nornir + NAPALM in parallel to run a compliance audit afterward, confirming every switch matches the expected state — structured data makes comparison trivial.
3.Paramiko SFTP to pull and archive running configs of all 500 switches as proof of change.

---
## ✅ Lab 2 Summary

| Part | Tool | Key Capability |
|------|------|----------------|
| 1 | **Paramiko** | Raw SSH — exec_command, SFTP |
| 2 | **Netmiko** | Vendor-aware SSH — send_command, send_config_set |
| 3 | **NAPALM** | Structured getters, diff-before-commit, rollback |
| 4 | **Nornir** | Inventory + parallel task execution |
| 5 | **Ansible** | Declarative YAML playbooks + Jinja2 templates |
| 6 | **REST APIs** | IP geolocation, BGP, RDAP, DNS-over-HTTPS |

### Tool Selection Quick Guide
```
SSH to one device quickly?            → Paramiko
SSH to many vendors?                  → Netmiko
Need structured data / safe commits?  → NAPALM
Parallel tasks + inventory mgmt?      → Nornir
Desired-state declarative config?     → Ansible
Live network intelligence?            → REST APIs
Production pipelines combine all six.
```

### What's Next?
- **GNS3 / EVE-NG** — Run this lab against real virtual Cisco/Juniper devices
- **Cisco DevNet Sandbox** — Free real device access: developer.cisco.com
- **Terraform** — Infrastructure-as-code for cloud network resources
- **CI/CD for Networks** — Git + GitHub Actions + Ansible for change automation

---
*ICS 3105: Multimedia Applications — Strathmore University*